# Darcy face-jump indicator

Equations (5.1)–(5.3) of Araya et al. (2013), [DOI 10.1137/120888223](https://doi.org/10.1137/120888223), measure broken pressure jumps. Exact local lifts are assumptions of the reliability theorem; this notebook uses finite P2 local spaces and does not infer a certified bound from the jump indicator alone.


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import TriangleMesh, solve_darcy
from pymhm.darcy_jump_estimator import estimate_darcy_jumps
root = Path.cwd()


In [ ]:
mesh = TriangleMesh.unit_square(2)
natural = {int(face): 0.0 for face in mesh.boundary_faces}
with threadpool_limits(1):
    solution = solve_darcy(mesh, degree=2, local_refinement=2, neumann=natural,
        source=lambda x: 8*np.pi**2*np.prod(np.cos(2*np.pi*x), axis=1), quadrature_order=10)
    indicator = estimate_darcy_jumps(solution, neumann=natural, calibration=3.0)
print({"jump_indicator": indicator.total, "macro_cells": len(mesh.cells)})


## Five-level numerical record

The archived sequences use n=4,8,16,32,64, four fine triangles per macro, and local/trace degree pairs (2,0) and (5,3). The paper's calibrations 3 and 50 are retained. The norm is a sum of a macro-diameter-weighted broken flux H(div) term, H-inverse pressure L2 error and broken gradient error. The constant-trace indicator lies below this finite-local error; the cubic-trace ratio tends to 1.684.


In [ ]:
rows = json.loads((root / "examples/results/darcy-jump/campaign.json").read_text())
for row in rows:
    print(row["trace_degree"], row["resolution"], row["paper_error_sum"], row["indicator"], row["effectivity"])
display(Image(filename=str(root / "docs/figures/darcy-jump/refinement.png")))
